In [10]:
N = 20  # for example

special_tokens = ["_", "[PAD]", "[UNK]", "[BOS]", "[EOS]"]
letters = [chr(i) for i in range(ord("a"), ord("z") + 1)]
numbers = [str(i) for i in range(0, N + 1)]
other_tokens = ["|", "?"]

all_tokens = special_tokens + letters + numbers + other_tokens

# Build vocab dict: token -> id
vocab = {tok: i for i, tok in enumerate(all_tokens)}
# vocab['_'] = -100
vocab

{'_': 0,
 '[PAD]': 1,
 '[UNK]': 2,
 '[BOS]': 3,
 '[EOS]': 4,
 'a': 5,
 'b': 6,
 'c': 7,
 'd': 8,
 'e': 9,
 'f': 10,
 'g': 11,
 'h': 12,
 'i': 13,
 'j': 14,
 'k': 15,
 'l': 16,
 'm': 17,
 'n': 18,
 'o': 19,
 'p': 20,
 'q': 21,
 'r': 22,
 's': 23,
 't': 24,
 'u': 25,
 'v': 26,
 'w': 27,
 'x': 28,
 'y': 29,
 'z': 30,
 '0': 31,
 '1': 32,
 '2': 33,
 '3': 34,
 '4': 35,
 '5': 36,
 '6': 37,
 '7': 38,
 '8': 39,
 '9': 40,
 '10': 41,
 '11': 42,
 '12': 43,
 '13': 44,
 '14': 45,
 '15': 46,
 '16': 47,
 '17': 48,
 '18': 49,
 '19': 50,
 '20': 51,
 '|': 52,
 '?': 53}

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase, NFD, StripAccents, Sequence as NormSequence

# 1. Create the internal tokenizer
wordlevel = WordLevel(vocab=vocab, unk_token="[UNK]")
tokenizer_backend = Tokenizer(wordlevel)

# 2. Make sure everything is lowercased (optional if your data is already clean)
tokenizer_backend.normalizer = NormSequence([
    NFD(),         # decompose accents
    StripAccents(),
    Lowercase(),
])

# 3. Split by spaces
tokenizer_backend.pre_tokenizer = Whitespace()

from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[BOS]",
    eos_token="[EOS]",
    mask_token="_",
)

# Sanity check
example = "3 c 15 a 2 b | _ _ _ 15 a 2 b 3"
encoded = hf_tokenizer(example)
print(encoded)
print(hf_tokenizer.convert_ids_to_tokens(encoded["input_ids"]))


{'input_ids': [34, 7, 46, 5, 33, 6, 52, 0, 0, 0, 46, 5, 33, 6, 34], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['3', 'c', '15', 'a', '2', 'b', '|', '_', '_', '_', '15', 'a', '2', 'b', '3']


In [21]:
hf_tokenizer.mask_token_id

0

In [18]:
import torch
from typing import Any, Dict, List
import numpy as np

def tensorize(example: Dict[str, Any]) -> Dict[str, Any]:
    tensorized = {}
    for key in ['input_ids', 'cu_seqlens']:
        if key not in example:
            continue
        if isinstance(example[key], List):
            tensorized[key] = torch.tensor(example[key], dtype=torch.long)
        elif isinstance(example[key], np.ndarray):
            tensorized[key] = torch.from_numpy(example[key])
        else:
            tensorized[key] = example[key]
    return tensorized

example = tensorize(encoded)
print(example)

{'input_ids': tensor([-100,    7,   46,    5,   33,    6,   52,    0,    0,    0,   46,    5,
          33,    6,   34])}


In [20]:
example['input_ids'][example['input_ids'] == 0] = -100  # should be -100
print(example)

{'input_ids': tensor([-100,    7,   46,    5,   33,    6,   52, -100, -100, -100,   46,    5,
          33,    6,   34])}


In [17]:
encoded['input_ids'][encoded['input_ids'] == 0] = -100  # should be -100
print(encoded)

{'input_ids': [-100, 7, 46, 5, 33, 6, 52, 0, 0, 0, 46, 5, 33, 6, 34], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [12]:
save_dir = "./my_synthetic_tokenizer"
hf_tokenizer.save_pretrained(save_dir)

# Later:
from transformers import PreTrainedTokenizerFast
tokenizer = PreTrainedTokenizerFast.from_pretrained(save_dir)
